In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torchsig.transforms.functional as F


OUTDIR = Path("ripple_plots")
OUTDIR.mkdir(exist_ok=True)

SEED = 12345

TAP_COUNTS = [65, 255, 1025, 2047]

MAX_RIPPLE_VALUES = [0.1, 0.2, 0.5, 1.0, 2.0, 3.0]
RIPPLE_FREQ_VALUES = [1.0, 2.0, 3.0, 5.0, 10.0]

MAG_Y_LIM = (-6, 1)

# Used only to identify the central passband for phase plotting.
# This avoids plotting phase where the magnitude response is near zero.
PHASE_PASSBAND_FLOOR_DB = -6.0


def ripple_response(
    num_taps: int,
    max_ripple_db: float = 1.0,
    ripple_freq: float = 3.0,
):
    """Return native-resolution frequency response of passband_ripple."""
    if num_taps % 2 == 0:
        num_taps += 1

    impulse = np.zeros(num_taps, dtype=np.float64)
    impulse[num_taps // 2] = 1.0

    y = F.passband_ripple(
        impulse,
        num_taps=num_taps,
        max_ripple_db=max_ripple_db,
        ripple_freq=ripple_freq,
        rng=np.random.default_rng(SEED),

        # Smooth phase makes the approximately linear phase response visible.
        passband_fuzz="random",
        stopband_fuzz="random",

        fallback="raise",
    )

    # Native filter-resolution response: no zero padding.
    H = np.fft.fftshift(np.fft.fft(y, n=num_taps))
    freq = np.fft.fftshift(np.fft.fftfreq(num_taps, d=1.0))

    mag_db = 20.0 * np.log10(np.maximum(np.abs(H), 1e-12))
    mag_db -= np.max(mag_db)

    return freq, H, mag_db


def central_passband_mask(freq, mag_db, floor_db=PHASE_PASSBAND_FLOOR_DB):
    """
    Return the contiguous magnitude-above-threshold region around DC.

    This keeps the phase plot focused on the passband and avoids noisy
    unwrapped phase in deep stopband regions.
    """
    candidate = mag_db >= floor_db

    dc_idx = int(np.argmin(np.abs(freq)))

    if not candidate[dc_idx]:
        raise RuntimeError(
            "Could not identify passband around DC. "
            "Try lowering PHASE_PASSBAND_FLOOR_DB."
        )

    left = dc_idx
    while left > 0 and candidate[left - 1]:
        left -= 1

    right = dc_idx
    while right < len(candidate) - 1 and candidate[right + 1]:
        right += 1

    mask = np.zeros_like(candidate, dtype=bool)
    mask[left:right + 1] = True

    return mask


def passband_unwrapped_phase(freq, H, mag_db):
    """Return unwrapped phase only over the central passband."""
    mask = central_passband_mask(freq, mag_db)

    f_pb = freq[mask]
    phase_pb = np.unwrap(np.angle(H[mask]))

    # Anchor the phase at DC so overlays are easier to compare.
    dc_idx = int(np.argmin(np.abs(f_pb)))
    phase_pb -= phase_pb[dc_idx]

    return f_pb, phase_pb


def save_magnitude_overlay_plot(
    traces,
    title: str,
    outfile: Path,
):
    plt.figure(figsize=(10, 5))

    for label, num_taps, max_ripple_db, ripple_freq in traces:
        freq, H, mag_db = ripple_response(
            num_taps=num_taps,
            max_ripple_db=max_ripple_db,
            ripple_freq=ripple_freq,
        )

        plt.plot(
            freq,
            mag_db,
            linewidth=1.0,
            marker=".",
            markersize=2,
            label=label,
        )

    plt.title(title)
    plt.xlabel("Normalized frequency")
    plt.ylabel("Magnitude response, normalized dB")
    plt.xlim(-0.5, 0.5)

    if MAG_Y_LIM is not None:
        plt.ylim(*MAG_Y_LIM)

    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outfile, dpi=200)
    plt.close()

    print(f"Saved {outfile}")


def save_phase_overlay_plot(
    traces,
    title: str,
    outfile: Path,
):
    plt.figure(figsize=(10, 5))

    for label, num_taps, max_ripple_db, ripple_freq in traces:
        freq, H, mag_db = ripple_response(
            num_taps=num_taps,
            max_ripple_db=max_ripple_db,
            ripple_freq=ripple_freq,
        )

        f_pb, phase_pb = passband_unwrapped_phase(freq, H, mag_db)

        line = plt.plot(
            f_pb,
            phase_pb,
            linewidth=1.0,
            marker=".",
            markersize=2,
            label=label,
        )[0]

        # Optional: overlay a least-squares linear phase fit.
        # This makes the approximate linearity easier to see.
        slope, intercept = np.polyfit(f_pb, phase_pb, deg=1)
        phase_fit = slope * f_pb + intercept

        plt.plot(
            f_pb,
            phase_fit,
            linestyle="--",
            linewidth=0.8,
            alpha=0.65,
            color=line.get_color(),
            label="_nolegend_",
        )

    plt.title(title)
    plt.xlabel("Normalized frequency")
    plt.ylabel("Unwrapped phase in passband, radians")
    plt.grid(True, alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outfile, dpi=200)
    plt.close()

    print(f"Saved {outfile}")


for num_taps in TAP_COUNTS:
    # Pair 1: fixed ripple_freq = 5.0, sweep max_ripple_db
    max_ripple_traces = [
        (
            f"max_ripple_db={max_ripple_db}",
            num_taps,
            max_ripple_db,
            5.0,
        )
        for max_ripple_db in MAX_RIPPLE_VALUES
    ]

    save_magnitude_overlay_plot(
        traces=max_ripple_traces,
        title=(
            "Passband ripple magnitude sweep: "
            f"num_taps = {num_taps}, fixed ripple_freq = 5.0"
        ),
        outfile=OUTDIR / (
            f"passband_ripple_magnitude_sweep_max_ripple_db_num_taps_{num_taps}.png"
        ),
    )

    save_phase_overlay_plot(
        traces=max_ripple_traces,
        title=(
            "Passband ripple unwrapped phase sweep: "
            f"num_taps = {num_taps}, fixed ripple_freq = 5.0"
        ),
        outfile=OUTDIR / (
            f"passband_ripple_phase_sweep_max_ripple_db_num_taps_{num_taps}.png"
        ),
    )

    # Pair 2: fixed max_ripple_db = 1.0, sweep ripple_freq
    ripple_freq_traces = [
        (
            f"ripple_freq={ripple_freq}",
            num_taps,
            1.0,
            ripple_freq,
        )
        for ripple_freq in RIPPLE_FREQ_VALUES
    ]

    save_magnitude_overlay_plot(
        traces=ripple_freq_traces,
        title=(
            "Passband ripple magnitude sweep: "
            f"num_taps = {num_taps}, fixed max_ripple_db = 1.0"
        ),
        outfile=OUTDIR / (
            f"passband_ripple_magnitude_sweep_ripple_freq_num_taps_{num_taps}.png"
        ),
    )

    save_phase_overlay_plot(
        traces=ripple_freq_traces,
        title=(
            "Passband ripple unwrapped phase sweep: "
            f"num_taps = {num_taps}, fixed max_ripple_db = 1.0"
        ),
        outfile=OUTDIR / (
            f"passband_ripple_phase_sweep_ripple_freq_num_taps_{num_taps}.png"
        ),
    )